# 🐼 pandas 한 바퀴 둘러보기
직원 데이터로 pandas 의 자주 쓰는 함수들을 하나씩 실행해 봅니다.
PyRewind 로 실행하면 **DataFrame 이 어떻게 바뀌는지** 단계별로 볼 수 있어요.

## 1. 데이터 만들기

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np

In [ ]:
data = {
    "이름": ["김민수", "이지은", "박서준", "최유나", "정우성", "한소희", "오세훈", "윤아름"],
    "부서": ["개발", "마케팅", "개발", "인사", "개발", "마케팅", "인사", "개발"],
    "나이": [28, 34, 45, 31, 39, 26, 52, 33],
    "연봉": [4200, 5100, 7800, 4600, 6300, 3900, 8200, 5500],
    "입사일": ["2021-03-02", "2018-07-15", "2010-01-04", "2019-11-20",
              "2014-05-12", "2023-02-01", "2005-09-30", "2017-08-21"],
    "평점": [4.1, 3.8, np.nan, 4.5, 3.2, np.nan, 4.8, 3.9],
}
df = pd.DataFrame(data)

In [ ]:
df.head(3)

## 2. 구조 살펴보기

In [ ]:
rows, cols = df.shape
col_names = list(df.columns)
print(rows, "행", cols, "열")

In [ ]:
dtypes = df.dtypes
dtypes

In [ ]:
stats = df.describe()
stats

## 3. 열 / 행 선택

In [ ]:
ages = df["나이"]
avg_age = ages.mean()
print("평균 나이:", avg_age)

In [ ]:
sub = df[["이름", "연봉"]]

In [ ]:
row2 = df.loc[2]
salary2 = df.loc[2, "연봉"]

In [ ]:
block = df.iloc[1:4, 0:3]

## 4. 조건으로 거르기

In [ ]:
high = df[df["연봉"] >= 6000]

In [ ]:
young_dev = df[(df["부서"] == "개발") & (df["나이"] < 35)]

In [ ]:
seniors = df.query("나이 >= 40 and 연봉 > 7000")

## 5. 새 열 만들기

In [ ]:
df["월급"] = (df["연봉"] / 12).round(1)

In [ ]:
def grade(score):
    if pd.isna(score):
        return "미평가"
    if score >= 4.5:
        return "A"
    elif score >= 4.0:
        return "B"
    return "C"

df["등급"] = df["평점"].apply(grade)

In [ ]:
dept_code = {"개발": "DEV", "마케팅": "MKT", "인사": "HR"}
df["부서코드"] = df["부서"].map(dept_code)

In [ ]:
df["연령대"] = pd.cut(df["나이"], bins=[20, 30, 40, 60], labels=["20대", "30대", "40대 이상"])

## 6. 결측치 처리

In [ ]:
missing = df.isna().sum()
missing

In [ ]:
mean_score = df["평점"].mean()
df["평점"] = df["평점"].fillna(round(mean_score, 2))

## 7. 정렬 · 개수 세기

In [ ]:
ranked = df.sort_values("연봉", ascending=False).head(5)

In [ ]:
dept_counts = df["부서"].value_counts()
dept_counts

## 8. 그룹별 집계

In [ ]:
dept_avg = df.groupby("부서")["연봉"].mean()

In [ ]:
summary = df.groupby("부서").agg(
    인원=("이름", "count"),
    평균연봉=("연봉", "mean"),
    최고나이=("나이", "max"),
)
summary

## 9. 날짜 · 문자열

In [ ]:
df["입사일"] = pd.to_datetime(df["입사일"])
df["입사연도"] = df["입사일"].dt.year
df["근속년수"] = 2026 - df["입사연도"]

In [ ]:
df["성"] = df["이름"].str[0]
kim_or_park = df[df["성"].isin(["김", "박"])]

## 10. 합치기

In [ ]:
bonus = pd.DataFrame({"부서": ["개발", "마케팅", "인사"], "보너스율": [0.10, 0.15, 0.05]})
merged = df[["이름", "부서", "연봉"]].merge(bonus, on="부서", how="left")
merged["보너스"] = merged["연봉"] * merged["보너스율"]

In [ ]:
new_people = pd.DataFrame({"이름": ["신입1", "신입2"], "부서": ["개발", "인사"], "나이": [24, 25]})
everyone = pd.concat([df[["이름", "부서", "나이"]], new_people], ignore_index=True)

## 11. 모양 바꾸기 (pivot / melt)

In [ ]:
sales = pd.DataFrame({
    "월": [1, 1, 2, 2, 3, 3],
    "지점": ["강남", "홍대", "강남", "홍대", "강남", "홍대"],
    "매출": [120, 90, 150, 80, 170, 110],
})
pivot = sales.pivot_table(index="월", columns="지점", values="매출", aggfunc="sum")

In [ ]:
long = pivot.reset_index().melt(id_vars="월", var_name="지점", value_name="매출")

In [ ]:
monthly = sales.groupby("월")["매출"].sum()
cum = monthly.cumsum()
moving = monthly.rolling(2).mean()

## 12. 한 줄씩 돌면서 계산하기 · 내보내기

In [ ]:
total_bonus = 0
for idx, row in merged.iterrows():
    total_bonus += row["보너스"]
print("총 보너스:", total_bonus)

In [ ]:
slim = df.drop(columns=["부서코드", "성"]).rename(columns={"연봉": "연봉(만원)"})
records = slim[["이름", "등급"]].head(3).to_dict("records")
records